In [ ]:

# 예시: Apple(AAPL) 2023년 일별 데이터
ticker = "AAPL"
df = yf.download(ticker, start="2023-01-01", end="2023-12-31")
df.reset_index(inplace=True)
df['Ticker'] = ticker

# 테이블 생성 (이미 존재하면 건너뜀)
cursor.execute(
    """
    BEGIN
        EXECUTE IMMEDIATE 'CREATE TABLE stock_data (
            ticker VARCHAR2(10),
            trade_date DATE,
            open_price NUMBER,
            high_price NUMBER,
            low_price NUMBER,
            close_price NUMBER,
            adj_close NUMBER,
            volume NUMBER
        )';
    EXCEPTION
        WHEN OTHERS THEN
            IF SQLCODE != -955 THEN RAISE; END IF;  -- ORA-00955: name is already used by an existing object
    END;
    """
)

# 데이터 삽입
insert_sql = (
    "INSERT INTO stock_data (ticker, trade_date, open_price, high_price, low_price, close_price, adj_close, volume)"
    " VALUES (:1, :2, :3, :4, :5, :6, :7, :8)"
)
rows = [(
    row['Ticker'],
    row['Date'],
    row['Open'],
    row['High'],
    row['Low'],
    row['Close'],
    row['Adj Close'],
    row['Volume']
) for _, row in df.iterrows()]

cursor.executemany(insert_sql, rows)
conn.commit()
cursor.close()
conn.close()

print("Data ingestion complete!")

In [ ]:
# streamlit_app.py
# Streamlit으로 Oracle DB 데이터를 시각화하는 대시보드

import streamlit as st
import pandas as pd
import oracledb

# Oracle 접속 정보 (ingestion.py와 동일)
USER = "your_username"
PASSWORD = "your_password"
HOST = "your_host"
PORT = 1521
SERVICE = "your_service_name"

dsn = oracledb.makedsn(HOST, PORT, service_name=SERVICE)
conn = oracledb.connect(user=USER, password=PASSWORD, dsn=dsn)

# 사이드바 설정
st.sidebar.title("설정")
tickers = st.sidebar.multiselect(
    "티커 선택", ["AAPL", "MSFT", "GOOG"], default=["AAPL"]
)
start_date = st.sidebar.date_input("시작일", pd.to_datetime("2023-01-01"))
end_date = st.sidebar.date_input("종료일", pd.to_datetime("2023-12-31"))

st.title("주식 가격 대시보드")

if tickers:
    # SQL문 생성
    in_clause = ",".join([f"'{t}'" for t in tickers])
    query = f"""
        SELECT ticker, trade_date, close_price
        FROM stock_data
        WHERE ticker IN ({in_clause})
        AND trade_date BETWEEN :start AND :end
        ORDER BY trade_date
    """
    df = pd.read_sql(query, conn, params={"start": start_date, "end": end_date})
    df['trade_date'] = pd.to_datetime(df['trade_date'])

    # 피벗하여 라인 차트 생성
    pivot_df = df.pivot(index='trade_date', columns='ticker', values='close_price')
    st.line_chart(pivot_df)
else:
    st.info("하나 이상의 티커를 선택하세요.")

conn.close()

# 실행 방법:
# 1) pip install yfinance oracledb pandas streamlit
# 2) python ingestion.py
# 3) streamlit run streamlit_app.py
